<h1>Chapter 6 - Native ReAct</h1>
<i>More Autonomy for your `TinyAgent`</i>


<a href="https://www.amazon.com/Illustrated-Guide-AI-Agents-Concepts/dp/B0GTYL2QSJ"><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="https://www.oreilly.com/library/view/an-illustrated-guide/9798341662681/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/HandsOnLLM/An-Illustrated-Guide-To-AI-Agents"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HandsOnLLM/An-Illustrated-Guide-To-AI-Agents/blob/main/chapter06/chapter06_native_react.ipynb)

---

This notebook is for Chapter 6 of [An Illustrated Guide to AI Agents](https://www.amazon.com/Illustrated-Guide-AI-Agents-Concepts/dp/B0GTYL2QSJ) by [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/) and [Jay Alammar](https://www.linkedin.com/in/jalammar).

---

<a href="https://www.amazon.com/Illustrated-Guide-AI-Agents-Concepts/dp/B0GTYL2QSJ">
<img src="https://learning.oreilly.com/covers/urn:orm:book:9798341662681/400w/" width="350"/></a>


### **[OPTIONAL]** - Installing Packages on Google Colab <img src="https://upload.wikimedia.org/wikipedia/commons/d/d0/Google_Colaboratory_SVG_Logo.svg" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** one of the following codeblock to install the dependencies for this chapter. If you want to use a cloud provider, you only need to run the following code block:

In [1]:
# %%capture
# !pip install illustrated-agents

---

💡 **NOTE**: If you want to use the GPU with `ollama`, then you will have to select a GPU first. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**. 

Then, **uncomment** and run this codeblock:

---

In [2]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma4:e4b &

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

## 1 - Choosing Your LLM - `Gemma 4`

At the beginning of every chapter, we start by choosing the LLM that we want to use. In this notebook, we will explore how to enable autonomous behavior for LLMs that have native tool calling and reasoning behavior. As such, the model that we will be using throughout this chapter is Gemma 4, a model with native tool calling and reasoning capabilities.

In [3]:
from illustrated_agents.chapters.ch2 import LLM

# Gemma 4 E4B (with native thinking and tool calling)
llm = LLM(model="gemma4:e4b", think=True)

If you want to use another LLM, here are a couple of options (both locally and on the cloud) that you can try:

In [4]:
# # llama.cpp
# llm = LLM(model="gemma-4-e4B-it-Q4_K_M", base_url="http://127.0.0.1:8080")

# # LM Studio
# llm = LLM(model="gemma-4-e4b-it", base_url="http://127.0.0.1:1234/v1")

# # OpenRouter
# import os
# llm = LLM(model="google/gemma-4-31b-it", base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])

## 2 - What does **`Autonomy`** look with Native Tool Calling and Reasoning?

In the previous chapter, we explored how to give your `TinyAgent` autonomy by explicit chains of `THOUGHT`, `OBSERVATION`, and `ACTION`. This worked quite well with a model that wasn't specifically trained for such autonomous behavior. We saw that this could be brittle needing to use regular expressions to extract those steps and have the underlying model execute actions. The parsing with JSON and XML, although a nice educational way of learning how it is done under the hood, is error-prone. 

Fortunately, we can use more recent models that were trained specifically for these tasks (reasoning and tool calling) as we did with native tool calling in Chapter 5 and native reasoning in Chapter 3. Let's explore how we can use newer models that have been trained to perform Reason and Act implicitly rather than explicitly as we did in `chapter06.ipynb`!

We previously implemented loops of:

* `THOUGHT` - A reasoning step about the current situation
* `ACTION` - An action to execute (e.g., a tool)
* `OBSERVATION` - A generated observation (typically the output of a tool)

However, with native tool calling there is no need for explicit `ACTION` and with native reasoning there is no need for explicit `THOUGHT`. We can replace what we already have for each of them with the following:

* `THOUGHT` -> Replace with `Response.reasoning` and `NativeReAct`
* `ACTION` -> Replace with `Response.tool_call` and `NativeTools`
* `OBSERVATION` -> Replace with adding the output of a tool to memory with `memory.add("tool", observation)`

Let's explore how we can do that starting with `NativeReAct`:




## 3 - "Building" `NativeReAct`

Since we can replace the idea of explicit `THOUGHT` / `ACTION` / `OBSERVATION` with native capabilities, the `ReAct` we used to have is not necessary anymore. Even moreso, there is only one functionality that needs to remain and that is the maximum number of steps each run can take. There is no need to instruct the model on `THOUGHT` / `ACTION` / `OBSERVATION` nor is parsing of them necessary. As such, we can create a `NativeReAct` class that simply does no processing to the `Response` nor has any `.prompt` that we will need to use:

In [5]:
from illustrated_agents.chapters.ch6 import ReAct

class NativeReAct(ReAct):
    """ReAct using native LLM reasoning instead of text-based parsing."""

    @property
    def prompt(self) -> str:
        return ""

    def parse(self, response):
        return response

And that's it! Really, there is nothing more that we need to add to use native reasoning and tool calling. The `NativeReAct` and `NativeTools` already handle all the "not-so-heavy" lifting.

## 4 - The Native `TinyAgent`

Usage of your newly fully native `TinyAgent` is the same as before but we use `NativeTools` instead of `Tools` and `NativeReAct` instead of `ReAct`. Let's start by initializing the `TinyAgent` first:

In [6]:
from illustrated_agents.chapters.ch4 import Memory
from illustrated_agents.chapters.ch5 import NativeTools
from illustrated_agents.chapters.ch6 import TinyAgent

from illustrated_agents.toolbox import add, multiply, subtract

# Register tools
tools = NativeTools()
tools.add_tool("add", add, "add(a: str, b: str)")
tools.add_tool("subtract", subtract, "subtract(a: str, b: str)")
tools.add_tool("multiply", multiply, "multiply(a: str, b: str)")

# Memory
memory = Memory()

# ReAct
react = NativeReAct(max_steps=10)

# Create agent
agent = TinyAgent(llm=llm, tools=tools, memory=memory, planner=react)

Next up, we can run the same script we did as in `chapter06.ipynb`:

In [7]:
# Multi-step task with reasoning
agent.run("What is (4.6 + 6.685) x 4, and then subtract 3.14 from the result?")

'The final result is 42.0.'

What's interesting about this response is that it is much more involved which is not surprising considering Gemma 4 is significantly better than Gemma 3 and also trained to perform better agentic loops. That said, let's explore the differences in messages between a native tool calling / reasoning model and a non-native model:

In [8]:
from rich import print

print(agent.trajectory.runs)

[
    {
        'query': 'What is (4.6 + 6.685) x 4, and then subtract 3.14 from the result?',
        'steps': [
            Step(
                thought='The user wants to perform a sequence of arithmetic operations:\n1. Calculate the sum of 
4.6 and 6.685: `4.6 + 6.685`.\n2. Multiply the result from step 1 by 4: `(4.6 + 6.685) * 4`.\n3. Subtract 3.14 from
the result of step 2: `((4.6 + 6.685) * 4) - 3.14`.\n\nI have the following tools: `add`, `subtract`, 
`multiply`.\n\nStep 1: `add(a="4.6", b="6.685")`\nStep 2: `multiply(a=<result of Step 1>, b="4")`\nStep 3: 
`subtract(a=<result of Step 2>, b="3.14")`\n\nI should chain these calls.\n\nFirst, call `add`.',
                action={'tool': 'add', 'kwargs': {'a': '4.6', 'b': '6.685'}},
                observation='11.285',
                answer=None,
                metadata=None
            ),
            Step(
                thought='The user wants to perform a calculation: `(4.6 + 6.685) x 4 - 3.14`.\n\nStep 1: Calculate 
`4.6 + 6.685`. (Already done in the previous step, result is `11.285`).\nStep 2: Calculate `11.285 * 4`.\nStep 3: 
Calculate `(result from Step 2) - 3.14`.\n\nI need to use the `multiply` tool for Step 2, and then the `subtract` 
tool for Step 3.\n\nStep 2 call: `multiply(a: "11.285", b: "4")`',
                action={'tool': 'multiply', 'kwargs': {'a': '11.285', 'b': '4'}},
                observation='45.14',
                answer=None,
                metadata=None
            ),
            Step(
                thought='The user wants to calculate two operations:\n1. `(4.6 + 6.685)`\n2. Then multiply the 
result by 4.\n3. Finally, subtract 3.14 from the result of step 2.\n\nStep 1: Add 4.6 and 6.685.\nThe previous tool
call used `add{a: "4.6", b: "6.685"}` and the response was `11.285`. This is correct.\n\nStep 2: Multiply 11.285 by
4.\nThe previous tool call used `multiply{a: "11.285", b: "4"}` and the response was `45.14`. This is 
correct.\n\nStep 3: Subtract 3.14 from 45.14.\nI need to use the `subtract` tool now.\na = 45.14\nb = 3.14',
                action={'tool': 'subtract', 'kwargs': {'a': '45.14', 'b': '3.14'}},
                observation='42.0',
                answer=None,
                metadata=None
            ),
            Step(
                thought='The user wants to calculate a multi-step arithmetic expression: `((4.6 + 6.685) * 4) - 
3.14`.\n\nI have already executed the steps using the available tools:\n\n1.  **Calculate `4.6 + 6.685`**:\n    *  
Tool call: `add(a: "4.6", b: "6.685")`\n    *   Tool response: `11.285` (This is correct: 4.6 + 6.685 = 
11.285)\n\n2.  **Calculate `11.285 * 4`**:\n    *   Tool call: `multiply(a: "11.285", b: "4")`\n    *   Tool 
response: `45.14` (This is correct: 11.285 * 4 = 45.14)\n\n3.  **Calculate `45.14 - 3.14`**:\n    *   Tool call: 
`subtract(a: "45.14", b: "3.14")`\n    *   Tool response: `42.0` (This is correct: 45.14 - 3.14 = 42.0)\n\nThe 
final result is 42.0. I should now provide this final answer to the user.',
                action=None,
                observation=None,
                answer='The final result is 42.0.',
                metadata={'model': 'gemma4:e4b', 'prompt_tokens': 328, 'completion_tokens': 334}
            )
        ]
    }
]

In [9]:
from illustrated_agents.utils import TrajectoryViewer
TrajectoryViewer(agent.trajectory)

Let's check the memory to see how exactly the tools are being called:

In [10]:
print(agent.memory.messages)

[
    {'role': 'system', 'content': 'You are a helpful assistant.\n\n'},
    {'role': 'user', 'content': 'What is (4.6 + 6.685) x 4, and then subtract 3.14 from the result?'},
    {
        'role': 'assistant',
        'content': '',
        'tool_calls': [
            {
                'id': 'call_rjflwe0i',
                'index': 0,
                'type': 'function',
                'function': {'name': 'add', 'arguments': '{"a":"4.6","b":"6.685"}'}
            }
        ]
    },
    {'role': 'tool', 'content': '11.285'},
    {
        'role': 'assistant',
        'content': '',
        'tool_calls': [
            {
                'id': 'call_qznsyyyz',
                'index': 0,
                'type': 'function',
                'function': {'name': 'multiply', 'arguments': '{"a":"11.285","b":"4"}'}
            }
        ]
    },
    {'role': 'tool', 'content': '45.14'},
    {
        'role': 'assistant',
        'content': '',
        'tool_calls': [
            {
                'id': 'call_xpqxid4g',
                'index': 0,
                'type': 'function',
                'function': {'name': 'subtract', 'arguments': '{"a":"45.14","b":"3.14"}'}
            }
        ]
    },
    {'role': 'tool', 'content': '42.0'},
    {'role': 'assistant', 'content': 'The final result is 42.0.'}
]

Note how the model uses assistant uses an additional `tool_calls` field to call a specific tool with a set of arguments. Moreover, we add the output of the tool to a specific `tool` role so that the model understand the effect of its tool call, which is essentially the same as the `OBSERVATION` step in explicit ReAct:


```json
[
    ...
    {
        'role': 'assistant',
        'content': '',
        'tool_calls': [
            {
                'id': 'call_8sezthoj',
                'function': {'arguments': '{"a":"45.14","b":"3.14"}', 'name': 'subtract'},
                'type': 'function',
                'index': 0
            }
        ]
    },
    {'role': 'tool', 'content': '42.0'},
    ...
]
```

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

# What We Built

In this chapter, we went from an explicit `ReAct`-loop to an implicit one! Since we already had worked on native capabilities early on there were fortunately few changes that needed to be made:

In [11]:
from illustrated_agents.chapters.ch6 import what_we_built_native; what_we_built_native

╭───────────────────────────────────────────────── What We Built ─────────────────────────────────────────────────╮
│ TinyAgent                                                                                                       │
│ ├── agent.py                                                                                                    │
│ ├── llm.py                                                                                                      │
│ ├── memory.py                                                                                                   │
│ ├── planning.py   ← New (Added the `NativeReAct` class)                                                         │
│ ├── toolbox.py                                                                                                  │
│ ├── tools.py                                                                                                    │
│ └── trajectory.py                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

# What's Next

You now have an autonomous Agent! There is still much more to do fortunately to give it additional capabilities. Next up, however, is a chapter focused on something important, namely evaluation. It covers various ways that you can evaluate your LLM and Agent.